In [1]:
import os
from pathlib import Path

import sqlglot
from sqlglot import exp, expressions

from src.utils.file_utils import parse_file_name
from src.migration.decomposer import SqlDecomposer, DecomposerWriter
from src.migration.metadata import MetadataProcessor
from src.migration.generator import PySparkGenerator
from src.paths import *

In [2]:
USERNAME

'dungp'

In [3]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_r_mhbos_m_client_crs.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_mhbos_m_client.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()[source_name] if source_name in all_source_rules else all_source_rules['default']

print(f"File gốc tại: {input_file}")

File gốc tại: C:\Users\dungp\projects\datalake-script\dml\com\com_t_mhbos_m_client.sql


In [5]:
# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = SqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
writer = DecomposerWriter()
writer.write(decomposed_script, output_root / file_name)
print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

# ==========================================
# BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
# ==========================================
processor = MetadataProcessor(source_rules)

try:
    # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
    pipeline_config = processor.process(decomposed_script, input_file)

    # Ghi file YAML
    metadata_output_dir = output_root / file_name / "metadata"
    processor.write_yaml(pipeline_config, metadata_output_dir)

    print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
    print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
    print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['primary_key']['logical_primary_key']}")
    print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

except FileNotFoundError as e:
    print(f"❌ [Lỗi Bước 2]: {e}")
    print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model 3a
   -> Khóa (Key) nhận diện được: ['client_no']
   -> File YAML đã lưu tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\metadata\com_t_mhbos_m_client.yaml


In [6]:
print("==========================================")
print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
print("==========================================")

with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
    pipeline_config = yaml.safe_load(f)

generator = PySparkGenerator(source_rules, output_mode="simple")
ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)

print("🎉 Hoàn tất toàn bộ Pipeline!")

 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
Generated DDL at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\ddl\com_t_mhbos_m_client.sql
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_all
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_primary_identification_type
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_primary_identification_no
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_secondary_identification_type
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_secondary_identification_no
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_identification_info
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_dob_info
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_gender_info
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_customer_name_1
Reading pre-processing SQL from: com_temp_t_mhbos_m_client_customer_name_2
Reading pre-processing SQL from: com_temp_t_mhbos_m_clien

In [ ]:
pipeline_config["target_table_name"]

In [ ]:
dml_context["main_processing_sqls"]

In [ ]:

# Read pre_processing SQLs
pre_processing_sqls = []
for step in pipeline_config.get("pre_processing", []):
    if step.get("action") == "skip":
        continue
    step_file = Path(step["file"])
    if step_file.exists():
        pre_processing_sqls.append(step_file.read_text(encoding="utf-8"))